# Term and Temp Hiring Outcomes

Are term and temp jobs less likely to get filled than permanent ones, holding job type fixed?

Data: GSA's [Federal Hiring Assessments and Selection Outcome Dataset](https://catalog.data.gov/dataset/federal-hiring-assessments-and-selection-outcome-dataset), USAJOBS announcements from the 24 CFO Act agencies, opened FY2020 through Dec 31, 2023. A "selection" means a hiring manager picked someone; not necessarily an accepted offer.

**Unit of analysis: positions, not announcements.** Agencies often post the same job twice at once (a public delegated-examining announcement and an internal merit-promotion one) and select off only one. Counting "selection made off this announcement" makes each permanent twin look like a failure even when the job was filled, while term jobs are almost always posted once, publicly. So the main outcome here is whether the *position* was filled through any announcement (GSA's `Selection_Made_through_Separate_Announcement` flag, which is "Yes" whenever there is a selection off this announcement or a linked one), with twin announcements we can find in the data collapsed into one position. That flag only exists for USA Staffing announcements, so Monster announcements are left out of the main results. That drops Education, State, EPA, Transportation and GSA almost entirely (they hire through Monster), and roughly half of Commerce, Treasury, USAID, Homeland Security and HUD. The old per-announcement outcome is kept as a comparison.

GSA dropped temporary-only announcements from this dataset, so "temp" here means announcements listing Temporary as one of several appointment types.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyfixest as pf
import requests

pd.set_option("display.width", 200)
# Fixed-effect cells with a single listing carry no within-cell information; pyfixest drops them and warns each time.
warnings.filterwarnings("ignore", message=r".*singleton fixed effect\(s\) dropped")
# In some single-department models temp_mix doesn't vary within job cells and is dropped; term is what we report.
warnings.filterwarnings("ignore", message=r"(?s).*The following variables are dropped: \['temp_mix'\]")

In [2]:
BASE = "https://inventory.data.gov/dataset/4189591b-d78e-4ad3-9d75-582175ad0b3f/resource"
FILES = [
    f"{BASE}/ef9d8bb1-2b4f-4618-98d1-bf96deaa6503/download/hiring_selections_and_outcomes_public_data_set_as_of_dec_31_2023_fy2020_fy2021.xlsx",
    f"{BASE}/dc0a8624-0f45-4d39-97d7-2a6694810d19/download/hiring_selections_and_outcomes_public_data_set_as_of_dec_31_2023_fy2022_fy2023.xlsx",
    f"{BASE}/6de5e82a-3e16-491c-baf9-204c3b2f4767/download/hiring_selections_and_outcomes_public_data_set_as_of_dec_31_2023_fy2024.xlsx",
]
EXPECTED_ROWS = 1161779

# GSA counts an announcement toward selection outcomes only after it has been
# open for four months. The data runs through Dec 31, 2023.
LAST_MATURE_OPEN_DATE = "2023-08-31"

GROUPS = ["Permanent", "Term", "Multiple incl. Temporary"]

SPECS = {
    "raw": "{y} ~ term + temp_mix",
    "series x grade FE": "{y} ~ term + temp_mix | series_grade",
    "+ bureau, FY, ann. type FE": "{y} ~ term + temp_mix | series_grade + ab + fy + anntype",
    "series x grade x bureau FE + FY + ann. type": "{y} ~ term + temp_mix | sg_ab + fy + anntype",
    "previous + log(apps+1)": "{y} ~ term + temp_mix + np.log1p(apps) | sg_ab + fy + anntype",
}

## Download and load

Downloads the files into `gsa_data/` the first time.

In [3]:
def download(data_dir):
    """Download the GSA xlsx files that aren't already in data_dir."""
    data_dir.mkdir(exist_ok=True)
    for url in FILES:
        path = data_dir / url.rsplit("/", 1)[1]
        if path.exists():
            continue
        print(f"Downloading {path.name}")
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)


def load(data_dir):
    """Read and stack the yearly files."""
    frames = []
    for url in FILES:
        path = data_dir / url.rsplit("/", 1)[1]
        sheets = pd.read_excel(path, sheet_name=None, engine="calamine", dtype=str)
        if len(sheets) != 1:
            raise ValueError(f"Expected one sheet in {path.name}, found {list(sheets)}")
        frames.append(next(iter(sheets.values())))
    df = pd.concat(frames, ignore_index=True)
    if len(df) != EXPECTED_ROWS:
        raise ValueError(f"Expected {EXPECTED_ROWS} rows, got {len(df)}")
    return df

In [4]:
data_dir = Path("gsa_data")
download(data_dir)
raw = load(data_dir)
raw.shape

(1161779, 41)

## Prepare

GSA counts an announcement toward selection outcomes only after it has been open four months, so announcements opened after Aug 31, 2023 are dropped. Job type is occupational series x pay plan/grade; `sg_ab` adds agency/bureau.

`sel` = selection made off this announcement. `filled` = selection made off this announcement or a linked one (USA Staffing only; missing for Monster).

In [5]:
def appointment_group(appt_type, appt_list):
    """Collapse Appointment_Type / Appointment_Type_List into analysis groups."""
    if appt_type == "Permanent":
        return "Permanent"
    if isinstance(appt_type, str) and appt_type.startswith("Term"):
        return "Term"
    if appt_type == "Multiple Appointment Types":
        parts = [p.strip() for p in str(appt_list).split(",")]
        if "Temporary" in parts:
            return "Multiple incl. Temporary"
        if "Term" in parts:
            return "Multiple incl. Term (no temp)"
        return "Multiple other"
    return "Other (intermittent/seasonal/etc.)"


def prepare(df):
    """Drop immature announcements and build outcome, group, and fixed-effect columns."""
    df = df.copy()
    df["open"] = pd.to_datetime(df.Announcement_Open_Date)
    df = df[df.open <= LAST_MATURE_OPEN_DATE].copy()
    df["appt"] = [appointment_group(t, l) for t, l in zip(df.Appointment_Type, df.Appointment_Type_List)]
    df["sel"] = (df.Selection_Made_off_this_Job_Announcement == "Yes").astype(int)
    usas = df.Source == "USA_Staffing"
    df["filled"] = np.where(usas, (df.Selection_Made_through_Separate_Announcement == "Yes").astype(float), np.nan)
    if ((df.sel == 1) & usas & (df.filled == 0)).any():
        raise ValueError("Found USA Staffing announcements with a selection but filled == 0; the flag doesn't mean what we assume")
    df["series_grade"] = df.Job_Series_Number.str.strip("'") + "|" + df.Pay_Plan + "-" + df.Grade.str.strip("'")
    df["ab"] = df.Agency + "|" + df.Bureau.fillna("(none)")
    df["sg_ab"] = df.series_grade + "|" + df.ab
    df["fy"] = df.Announcement_Open_Date_FY
    df["anntype"] = df.Announcement_Type.str.upper().str.strip()
    df["apps"] = pd.to_numeric(df.Total_Applications, errors="coerce")
    df["public"] = df.anntype.str.contains("DE|DELEGATED|PUBLIC", regex=True).astype(int)
    df["term"] = (df.appt == "Term").astype(int)
    df["temp_mix"] = (df.appt == "Multiple incl. Temporary").astype(int)
    return df

In [6]:
df = prepare(raw)
len(df)

1061305

## Collapse twin announcements into positions

Twins get separate announcement numbers, so they're matched on agency, bureau, series, grade, job title, open and close dates, and locations. A position is filled if any of its announcements is flagged filled; applications are summed across twins. The few matched groups that mix appointment groups (e.g. a term and a permanent posting) are dropped, since they can't be assigned to one side. Many twins aren't in the public file at all (GSA removed direct-hire and excepted-service announcements), which is why the GSA flag rather than this matching is the outcome.

In [7]:
TWIN_KEY = ["Agency", "Bureau", "Job_Series_Number", "Grade", "title_n", "Announcement_Open_Date", "Announcement_Close_Date", "Announcement_Locations"]


def to_positions(df):
    """USA Staffing announcements collapsed to one row per position (twin announcements merged)."""
    u = df[df.Source == "USA_Staffing"].copy()
    u["title_n"] = u.Vacancy_Job_Title.str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True).str.split().str.join(" ")
    u["pos_id"] = u.groupby(TWIN_KEY, dropna=False).ngroup()
    mixed = u.groupby("pos_id").appt.transform("nunique") > 1
    print(f"USA Staffing announcements: {len(u)}; dropped {int(mixed.sum())} in {u[mixed].pos_id.nunique()} matched groups that mix appointment groups")
    u = u[~mixed]
    first_cols = ["Agency", "Bureau", "Job_Series_Number", "appt", "term", "temp_mix", "series_grade", "ab", "sg_ab", "fy", "open"]
    agg = {c: "first" for c in first_cols}
    agg.update(filled=("filled", "max"), sel=("sel", "max"), apps=("apps", "sum"), n_announcements=("sel", "size"),
               anntype=("anntype", lambda x: " + ".join(sorted(set(x)))))
    agg = {k: (v if isinstance(v, tuple) else (k, v)) for k, v in agg.items()}
    pos = u.groupby("pos_id").agg(**agg)
    print(f"Positions: {len(pos)}")
    return pos

In [8]:
pos = to_positions(df)

USA Staffing announcements: 989115; dropped 2040 in 971 matched groups that mix appointment groups


Positions: 922386


## Raw rates

Per announcement (all sources) vs per position (USA Staffing). `twinned` = share of positions posted as more than one announcement in the data.

In [9]:
def raw_rates(df, pos):
    """Selection rate per announcement and fill rate per position, by appointment group."""
    a = df.groupby("appt").agg(announcements=("sel", "size"), sel_per_announcement=("sel", "mean"), public_share=("public", "mean"))
    p = pos.groupby("appt").agg(positions=("filled", "size"), filled_per_position=("filled", "mean"),
                                twinned=("n_announcements", lambda s: (s > 1).mean()), median_apps=("apps", "median"))
    return a.join(p).sort_values("announcements", ascending=False)


raw_rates(df, pos).round(3)

,announcements,sel_per_announcement,public_share,positions,filled_per_position,twinned,median_apps
appt,,,,,,,
Permanent,1005987,0.568,0.659,871389,0.681,0.071,17.0
Term,33716,0.503,0.920,30951,0.593,0.038,17.0
Other (intermittent/seasonal/etc.),9553,0.504,0.924,9151,0.607,0.023,9.0
Multiple incl. Temporary,5903,0.443,0.764,5538,0.549,0.024,8.0
Multiple other,3816,0.631,0.662,3404,0.731,0.080,18.0
Multiple incl. Term (no temp),2330,0.456,0.961,1953,0.615,0.015,10.0


## Holding job type fixed

Linear probability models, standard errors clustered by bureau. `term` and `temp_mix` are percentage-point differences from permanent. The last spec adds log applications, which is likely part of the mechanism rather than a confounder.

Main result: `filled` per position. Comparison: `sel` per announcement (the original measure).

In [10]:
def regressions(df, pos):
    """Term / temp gaps under each spec, for the position-filled outcome and the per-announcement comparison."""
    rows = []
    for label, y, data in [("filled per position (main)", "filled", pos), ("sel per announcement (old)", "sel", df)]:
        m = data[data.appt.isin(GROUPS)]
        for name, f in SPECS.items():
            fit = pf.feols(f.format(y=y), data=m, vcov={"CRV1": "ab"})
            t = fit.tidy()
            for v in ["term", "temp_mix"]:
                rows.append(dict(outcome=label, spec=name, var=v, coef=t.loc[v, "Estimate"], se=t.loc[v, "Std. Error"], n=fit._N))
    res = pd.DataFrame(rows)
    res["ci_lo"] = res.coef - 1.96 * res.se
    res["ci_hi"] = res.coef + 1.96 * res.se
    return res


regressions(df, pos).round(4)

,outcome,spec,var,coef,se,n,ci_lo,ci_hi
0,filled per position (main),raw,term,-0.0874,0.0144,907878,-0.1156,-0.0593
1,filled per position (main),raw,temp_mix,-0.1323,0.0229,907878,-0.1772,-0.0874
2,filled per position (main),series x grade FE,term,-0.1011,0.0119,903059,-0.1245,-0.0777
3,filled per position (main),series x grade FE,temp_mix,-0.1094,0.0221,903059,-0.1527,-0.0661
4,filled per position (main),"+ bureau, FY, ann. type FE",term,-0.1088,0.0090,903051,-0.1264,-0.0912
5,filled per position (main),"+ bureau, FY, ann. type FE",temp_mix,-0.1239,0.0243,903051,-0.1716,-0.0763
6,filled per position (main),series x grade x bureau FE + FY + ann. type,term,-0.1148,0.0079,893493,-0.1302,-0.0994
7,filled per position (main),series x grade x bureau FE + FY + ann. type,temp_mix,-0.0985,0.0293,893493,-0.1560,-0.0410
8,filled per position (main),previous + log(apps+1),term,-0.0807,0.0090,893493,-0.0983,-0.0631
9,filled per position (main),previous + log(apps+1),temp_mix,-0.0658,0.0152,893493,-0.0957,-0.0360


## Matched check

Within series x grade x bureau x FY cells that have both term and permanent positions, weighted by term positions.

In [11]:
def matched_cells(pos):
    """Term-weighted term-minus-permanent fill-rate gap within series x grade x bureau x FY cells that have both."""
    c = pos[pos.appt.isin(["Permanent", "Term"])]
    cell = c.groupby(["sg_ab", "fy", "appt"]).filled.agg(["mean", "size"]).unstack("appt").dropna()
    w = cell[("size", "Term")]
    return {
        "cells": len(cell),
        "term_positions_matched": int(w.sum()),
        "term_positions_total": int((c.appt == "Term").sum()),
        "perm_rate": float(np.average(cell[("mean", "Permanent")], weights=w)),
        "term_rate": float(np.average(cell[("mean", "Term")], weights=w)),
    }


matched_cells(pos)

{'cells': 7172,
 'term_positions_matched': 27898,
 'term_positions_total': 30951,
 'perm_rate': 0.7064552528541441,
 'term_rate': 0.5888235715821922}

## By department, and how much is applicant pool size?

For each group: raw fill rates, the term gap (series x grade x bureau, FY, announcement type fixed effects), the gap after also controlling for log applications, and the share of the gap that goes away with that control. The share is only shown where the gap is statistically significant (negative or positive); for a gap near zero the ratio is meaningless. A negative share means the gap gets bigger once applications are held fixed. `old_gap` is the same model on the per-announcement outcome, for comparison.

Science/tech (STEM) = series 04xx (bio sciences), 08xx (engineering), 13xx (physical sciences), 15xx (math/stats/CS), 2210 (IT).

Pooled rows are clustered by bureau. Single-department rows are clustered by series x grade x bureau, since many departments have one or a few bureaus. Groups with fewer than 30 term positions in job cells that also have permanent positions are left blank.

In [12]:
MIN_IDENTIFIED_TERM = 30


def is_stem(series):
    """Science/tech series: 04xx bio sciences, 08xx engineering, 13xx physical sciences, 15xx math/stats/CS, 2210 IT."""
    s = series.str.strip("'")
    return s.str[:2].isin(["04", "08", "13", "15"]) | (s == "2210")


def identified_term(sub):
    """Term rows in series x grade x bureau cells that also have permanent rows (the ones the FE model compares)."""
    perm_cells = sub.loc[(sub.term == 0) & (sub.temp_mix == 0), "sg_ab"].unique()
    return int(((sub.term == 1) & sub.sg_ab.isin(perm_cells)).sum())


def term_gap(sub, y, cluster, control_apps):
    """Term coefficient from the main FE spec, optionally controlling for log applications."""
    x = "term + temp_mix" if sub.temp_mix.sum() > 0 else "term"
    if control_apps:
        x += " + np.log1p(apps)"
    t = pf.feols(f"{y} ~ {x} | sg_ab + fy + anntype", data=sub, vcov={"CRV1": cluster}).tidy()
    return t.loc["term", "Estimate"], t.loc["term", "Std. Error"]


def applicant_share_row(name, sub, old_sub, cluster):
    """Fill rates, term gap with and without the applications control, and the old per-announcement gap; all jobs and STEM."""
    rows = []
    for jobs, s, o in [("all jobs", sub, old_sub), ("STEM", sub[is_stem(sub.Job_Series_Number)], old_sub[is_stem(old_sub.Job_Series_Number)])]:
        perm = (s.term == 0) & (s.temp_mix == 0)
        r = dict(
            group=name,
            jobs=jobs,
            term_positions=int(s.term.sum()),
            term_comparable=identified_term(s),
            median_apps_term=s.loc[s.term == 1, "apps"].median(),
            median_apps_perm=s.loc[perm, "apps"].median(),
            fill_rate_term=s.loc[s.term == 1, "filled"].mean(),
            fill_rate_perm=s.loc[perm, "filled"].mean(),
        )
        if r["term_comparable"] >= MIN_IDENTIFIED_TERM:
            r["gap"], r["gap_se"] = term_gap(s, "filled", cluster, control_apps=False)
            r["gap_ctrl_apps"], r["gap_ctrl_apps_se"] = term_gap(s, "filled", cluster, control_apps=True)
        if identified_term(o) >= MIN_IDENTIFIED_TERM:
            r["old_gap"], r["old_gap_se"] = term_gap(o, "sel", cluster, control_apps=False)
        rows.append(r)
    return rows


def applicant_share(df, pos):
    """Department table for all departments, all except DoD, and each department."""
    m = pos[pos.appt.isin(GROUPS)]
    o = df[df.appt.isin(GROUPS)]
    dod = "Department of Defense"
    rows = applicant_share_row("All departments", m, o, "ab")
    rows += applicant_share_row("All except DoD", m[m.Agency != dod], o[o.Agency != dod], "ab")
    for agency in sorted(o.Agency.unique()):
        rows += applicant_share_row(agency, m[m.Agency == agency], o[o.Agency == agency], "sg_ab")
    res = pd.DataFrame(rows)
    res["sig_negative"] = res.gap + 1.96 * res.gap_se < 0
    res["sig_positive"] = res.gap - 1.96 * res.gap_se > 0
    res["share_explained_by_apps"] = (1 - res.gap_ctrl_apps / res.gap).where(res.sig_negative | res.sig_positive)
    return res


shares = applicant_share(df, pos)
shares.round(3)

,group,jobs,term_positions,term_comparable,median_apps_term,median_apps_perm,fill_rate_term,fill_rate_perm,gap,gap_se,gap_ctrl_apps,gap_ctrl_apps_se,old_gap,old_gap_se,sig_negative,sig_positive,share_explained_by_apps
0,All departments,all jobs,30951,29248,17.0,17.0,0.593,0.681,-0.115,0.008,-0.081,0.009,-0.107,0.013,True,False,0.297
1,All departments,STEM,6578,5757,19.0,16.0,0.602,0.629,-0.098,0.020,-0.055,0.016,-0.082,0.034,True,False,0.444
2,All except DoD,all jobs,8174,7031,29.0,14.0,0.635,0.687,-0.088,0.021,-0.078,0.021,-0.034,0.034,True,False,0.118
3,All except DoD,STEM,3189,2585,35.0,20.0,0.707,0.686,-0.026,0.022,-0.010,0.019,0.037,0.030,False,False,NaN
4,Department of Agriculture,all jobs,1188,1065,30.0,18.0,0.699,0.718,-0.022,0.019,0.004,0.018,-0.000,0.019,False,False,NaN
5,Department of Agriculture,STEM,812,750,31.0,18.0,0.725,0.726,0.004,0.021,0.028,0.020,0.025,0.021,False,False,NaN
6,Department of Commerce,all jobs,367,281,26.0,59.0,0.670,0.749,-0.090,0.040,-0.064,0.040,0.027,0.031,True,False,0.283
7,Department of Commerce,STEM,228,184,23.0,41.0,0.737,0.759,-0.006,0.041,0.016,0.042,0.068,0.035,False,False,NaN
8,Department of Defense,all jobs,22777,22217,15.0,20.0,0.579,0.673,-0.122,0.005,-0.073,0.005,-0.127,0.007,True,False,0.398
9,Department of Defense,STEM,3389,3172,11.0,15.0,0.503,0.595,-0.135,0.015,-0.073,0.015,-0.146,0.015,True,False,0.459


### How many departments show a term penalty?

In [13]:
def count_departments(shares):
    """Among single departments with enough comparable term listings: significantly negative, positive, or neither."""
    d = shares[~shares.group.isin(["All departments", "All except DoD"]) & shares.gap.notna()]
    out = d.groupby("jobs").agg(
        departments=("group", "size"),
        sig_negative=("sig_negative", "sum"),
        sig_positive=("sig_positive", "sum"),
    )
    out["not_significant"] = out.departments - out.sig_negative - out.sig_positive
    neg = d[d.sig_negative].groupby("jobs").group.apply(list)
    pos = d[d.sig_positive].groupby("jobs").group.apply(list)
    return out.assign(negative=neg, positive=pos)


count_departments(shares)

,departments,sig_negative,sig_positive,not_significant,negative,positive
jobs,,,,,,
STEM,7,3,0,4,"[Department of Defense, Department of Veterans...",NaN
all jobs,13,8,0,5,"[Department of Commerce, Department of Defense...",NaN
